In [ ]:
import numpy as np
import pandas as pd
import time

In [ ]:
RNG = np.random.default_rng(42)
CSV_PATH = "creditcard.csv"
SAMPLE_SIZE = 5000
def train_test_split(X, y, test_size=0.2):
    X, y = np.asarray(X), np.asarray(y)
    test_idx, train_idx = [], []
    for cls in np.unique(y):
        idx = np.where(y == cls)[0]
        RNG.shuffle(idx)
        n_test = int(len(idx) * test_size)
        test_idx.extend(idx[:n_test])
        train_idx.extend(idx[n_test:])
    train_idx, test_idx = np.array(train_idx), np.array(test_idx)
    RNG.shuffle(train_idx)
    RNG.shuffle(test_idx)
    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]

class StandardScaler:
    def fit(self, X):
        self.mean_ = X.mean(axis=0)
        self.std_ = X.std(axis=0)
        self.std_[self.std_ == 0] = 1.0
        return self

    def transform(self, X):
        return (X - self.mean_) / self.std_

    def fit_transform(self, X):
        return self.fit(X).transform(X)

def smote(X, y, k=5):
    classes, counts = np.unique(y, return_counts=True)
    minority_class = classes[np.argmin(counts)]
    majority_count = counts.max()
    minority_count = counts.min()
    n_to_generate = majority_count - minority_count

    X_min = X[y == minority_class]
    n_min = len(X_min)
    k = min(k, n_min - 1) if n_min > 1 else 1

    synthetic = []
    for _ in range(n_to_generate):
        i = RNG.integers(0, n_min)
        dists = np.linalg.norm(X_min - X_min[i], axis=1)
        neighbor_idx = np.argsort(dists)[1:k + 1]  # exclude self (dist=0)
        nn = X_min[RNG.choice(neighbor_idx)]
        lam = RNG.uniform(0, 1)
        synthetic.append(X_min[i] + lam * (nn - X_min[i]))

    X_syn = np.array(synthetic) if synthetic else np.empty((0, X.shape[1]))
    y_syn = np.full(len(X_syn), minority_class)

    X_res = np.vstack([X, X_syn])
    y_res = np.concatenate([y, y_syn])
    shuffle_idx = RNG.permutation(len(X_res))
    return X_res[shuffle_idx], y_res[shuffle_idx]


class LogisticRegression:
    def __init__(self, lr=0.1, n_iters=1000, C=1.0):
        self.lr = lr
        self.n_iters = n_iters
        self.lam = 1.0 / C

    def _sigmoid(self, z):
        return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.w = np.zeros(n_features)
        self.b = 0.0
        for _ in range(self.n_iters):
            z = X @ self.w + self.b
            preds = self._sigmoid(z)
            error = preds - y
            dw = (X.T @ error) / n_samples + (self.lam / n_samples) * self.w
            db = error.mean()
            self.w -= self.lr * dw
            self.b -= self.lr * db
        return self

    def predict_proba(self, X):
        return self._sigmoid(X @ self.w + self.b)

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)


class _Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value 


class DecisionTree:
    def __init__(self, max_depth=10, min_samples_split=2, n_features=None):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.n_features = n_features

    def _gini(self, y):
        if len(y) == 0:
            return 0
        p = y.mean()
        return 1 - p**2 - (1 - p)**2

    def _best_split(self, X, y, feat_idxs):
        best_gain, best_feat, best_thresh = -1, None, None
        parent_gini = self._gini(y)
        for feat in feat_idxs:
            thresholds = np.unique(np.percentile(X[:, feat], np.linspace(5, 95, 19)))
            for t in thresholds:
                left_mask = X[:, feat] <= t
                if left_mask.sum() == 0 or (~left_mask).sum() == 0:
                    continue
                n = len(y)
                gl, gr = self._gini(y[left_mask]), self._gini(y[~left_mask])
                w = left_mask.sum() / n * gl + (~left_mask).sum() / n * gr
                gain = parent_gini - w
                if gain > best_gain:
                    best_gain, best_feat, best_thresh = gain, feat, t
        return best_feat, best_thresh, best_gain

    def _build(self, X, y, depth=0):
        n_samples, n_feats = X.shape
        if (depth >= self.max_depth or n_samples < self.min_samples_split
                or len(np.unique(y)) == 1):
            return _Node(value=y.mean() if n_samples else 0.0)

        feat_idxs = RNG.choice(n_feats, self.n_features, replace=False)
        feat, thresh, gain = self._best_split(X, y, feat_idxs)
        if feat is None or gain <= 0:
            return _Node(value=y.mean())

        left_mask = X[:, feat] <= thresh
        left = self._build(X[left_mask], y[left_mask], depth + 1)
        right = self._build(X[~left_mask], y[~left_mask], depth + 1)
        return _Node(feature=feat, threshold=thresh, left=left, right=right)

    def fit(self, X, y):
        if self.n_features is None:
            self.n_features = max(1, int(np.sqrt(X.shape[1])))
        self.root = self._build(X, y)
        return self

    def _predict_one(self, x, node):
        if node.value is not None:
            return node.value
        branch = node.left if x[node.feature] <= node.threshold else node.right
        return self._predict_one(x, branch)

    def predict_proba(self, X):
        return np.array([self._predict_one(x, self.root) for x in X])


class RandomForest:
    def __init__(self, n_trees=20, max_depth=10, min_samples_split=2):
        self.n_trees = n_trees
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split

    def fit(self, X, y):
        self.trees = []
        n_samples = X.shape[0]
        for _ in range(self.n_trees):
            idxs = RNG.integers(0, n_samples, n_samples)  # bootstrap sample
            tree = DecisionTree(self.max_depth, self.min_samples_split)
            tree.fit(X[idxs], y[idxs])
            self.trees.append(tree)
        return self

    def predict_proba(self, X):
        preds = np.array([t.predict_proba(X) for t in self.trees])
        return preds.mean(axis=0)

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)


def precision_recall(y_true, y_pred):
    tp = np.sum((y_pred == 1) & (y_true == 1))
    fp = np.sum((y_pred == 1) & (y_true == 0))
    fn = np.sum((y_pred == 0) & (y_true == 1))
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    return precision, recall

def roc_auc(y_true, y_scores):
    order = np.argsort(-y_scores)
    y_true = y_true[order]
    n_pos = y_true.sum()
    n_neg = len(y_true) - n_pos
    if n_pos == 0 or n_neg == 0:
        return 0.5
    tps = np.cumsum(y_true)
    fps = np.cumsum(1 - y_true)
    tpr = tps / n_pos
    fpr = fps / n_neg
    tpr, fpr = np.concatenate([[0], tpr]), np.concatenate([[0], fpr])
    return np.sum((fpr[1:] - fpr[:-1]) * (tpr[1:] + tpr[:-1]) / 2)  # trapezoidal rule


if __name__ == "__main__":
    t0 = time.time()
    df = pd.read_csv(creditcard.csv)

    if SAMPLE_SIZE is not None:
        fraud = df[df.Class == 1]
        normal = df[df.Class == 0].sample(n=SAMPLE_SIZE, random_state=42)
        df = pd.concat([fraud, normal]).reset_index(drop=True)
    print(f"Dataset shape: {df.shape}  (fraud rows: {(df.Class == 1).sum()})")

    X = df.drop("Class", axis=1).values
    y = df["Class"].values

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    X_train_sm, y_train_sm = smote(X_train_scaled, y_train)

    lr = LogisticRegression(lr=0.1, n_iters=500, C=1.0).fit(X_train_sm, y_train_sm)
    lr_preds = lr.predict(X_test_scaled)
    lr_probs = lr.predict_proba(X_test_scaled)
    p, r = precision_recall(y_test, lr_preds)
    print(f"Logistic Regression -> Precision: {p:.4f}  Recall: {r:.4f}  "
          f"ROC-AUC: {roc_auc(y_test, lr_probs):.4f}  (t={time.time()-t0:.1f}s)")

    X_train_sm_rf, y_train_sm_rf = smote(X_train, y_train)
    rf = RandomForest(n_trees=10, max_depth=6).fit(X_train_sm_rf, y_train_sm_rf)
    rf_preds = rf.predict(X_test)
    rf_probs = rf.predict_proba(X_test)
    p, r = precision_recall(y_test, rf_preds)
    print(f"Random Forest       -> Precision: {p:.4f}  Recall: {r:.4f}  "
          f"ROC-AUC: {roc_auc(y_test, rf_probs):.4f}  (t={time.time()-t0:.1f}s)")